In [2]:
import os
from databricks import sql
from dotenv import load_dotenv

In [3]:
# Carrega as variáveis de ambiente
load_dotenv()

server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME")
http_path = os.getenv("DATABRICKS_HTTP_PATH")
access_token = os.getenv("DATABRICKS_TOKEN")

print("Ambiente configurado e pronto para a Camada Gold.")

Ambiente configurado e pronto para a Camada Gold.


In [4]:
with sql.connect(
    server_hostname=server_hostname,
    http_path=http_path,
    access_token=access_token
) as connection:
    with connection.cursor() as cursor:
        print("Criando tabela fato da camada Gold ('cyclistic.gold.trips_fact')...")
        
        query_gold_fact = """
        CREATE OR REPLACE TABLE cyclistic.gold.trips_fact AS
        WITH base_calculos AS (
            SELECT
                ride_id,
                rideable_type,
                started_at,
                ended_at,
                member_casual,
                start_station_name,
                start_station_id,
                end_station_name,
                end_station_id,
                start_lat,
                start_lng,
                end_lat,
                end_lng,
                -- Duração em minutos
                ROUND((UNIX_TIMESTAMP(ended_at) - UNIX_TIMESTAMP(started_at)) / 60.0, 2) AS ride_length_minutes,
                
                -- Componentes de Data e Hora
                DATE(started_at) AS ride_date,
                YEAR(started_at) AS ride_year,
                MONTH(started_at) AS ride_month,
                DATE_FORMAT(started_at, 'MMMM') AS month_name,
                DAYOFWEEK(started_at) AS day_of_week, -- 1 = Domingo, 7 = Sábado
                DATE_FORMAT(started_at, 'EEEE') AS day_name,
                HOUR(started_at) AS start_hour,
                
                -- Período do dia
                CASE 
                    WHEN HOUR(started_at) BETWEEN 5 AND 11 THEN 'Manhã'
                    WHEN HOUR(started_at) BETWEEN 12 AND 17 THEN 'Tarde'
                    WHEN HOUR(started_at) BETWEEN 18 AND 22 THEN 'Noite'
                    ELSE 'Madrugada'
                END AS time_of_day,
                
                -- Conversão de coordenadas para Radianos para o Haversine
                RADIANS(start_lat) AS lat1_rad,
                RADIANS(end_lat) AS lat2_rad,
                RADIANS(end_lat - start_lat) AS dlat_rad,
                RADIANS(end_lng - start_lng) AS dlng_rad
            FROM cyclistic.silver.tripdata_clean
        ),
        haversine_calc AS (
            SELECT
                *,
                -- Fórmula de Haversine: Raio médio da Terra = 6371.0 km
                ROUND(
                    6371.0 * 2 * ATAN2(
                        SQRT(
                            SIN(dlat_rad / 2) * SIN(dlat_rad / 2) +
                            COS(lat1_rad) * COS(lat2_rad) * SIN(dlng_rad / 2) * SIN(dlng_rad / 2)
                        ),
                        SQRT(
                            1 - (
                                SIN(dlat_rad / 2) * SIN(dlat_rad / 2) +
                                COS(lat1_rad) * COS(lat2_rad) * SIN(dlng_rad / 2) * SIN(dlng_rad / 2)
                            )
                        )
                    ),
                    2
                ) AS distance_km
            FROM base_calculos
        )
        SELECT
            ride_id,
            rideable_type,
            started_at,
            ended_at,
            member_casual,
            ride_date,
            ride_year,
            ride_month,
            month_name,
            day_of_week,
            day_name,
            start_hour,
            time_of_day,
            ride_length_minutes,
            distance_km,
            start_station_name,
            start_station_id,
            end_station_name,
            end_station_id,
            start_lat,
            start_lng,
            end_lat,
            end_lng
        FROM haversine_calc;
        """
        
        cursor.execute(query_gold_fact)
        
        cursor.execute("SELECT COUNT(*) AS total FROM cyclistic.gold.trips_fact;")
        total_gold = cursor.fetchall()[0]['total']
        print(f"🎉 Tabela Gold 'trips_fact' criada com sucesso!")
        print(f"📊 Total de registros prontos para consumo: {total_gold:,}".replace(",", "."))

Criando tabela fato da camada Gold ('cyclistic.gold.trips_fact')...
🎉 Tabela Gold 'trips_fact' criada com sucesso!
📊 Total de registros prontos para consumo: 26.058.106
